# Инициализация

In [6]:
import os
import sys

from sedona.spark import SedonaContext

# Явно указываем путь к Python для воркеров
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Настройка Hadoop
os.environ['HADOOP_HOME'] = r'C:\Hadoop\hadoop-3.3.6'

# Пакеты для Spark 3.5.4 со Scala 2.12
additional_packages = [
    "org.apache.sedona:sedona-spark-3.5_2.12:1.8.0",
    "org.datasyslab:geotools-wrapper:1.8.0-33.1"
]

# Оптимальная конфигурация для 24 ядер
config = SedonaContext.builder() \
    .appName("SedonaApp") \
    .master("local[24]") \
    .config("spark.sql.shuffle.partitions", "48") \
    .config("spark.default.parallelism", "48") \
    .config("spark.task.cpus", "1") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryo.registrator", "org.apache.sedona.core.serde.SedonaKryoRegistrator") \
    .config("spark.sql.extensions", "org.apache.sedona.sql.SedonaSqlExtensions,org.apache.sedona.viz.sql.SedonaVizExtensions") \
    .config("spark.jars", r"D:\Artem\Work\amtech_projects\postgresql-42.7.13.jar") \
    .config("spark.jars.packages", ",".join(additional_packages)) \
    .config("spark.driver.memory", "16g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

# config = SedonaContext.builder() \
#     .appName("SedonaApp") \
#     .config("spark.jars.packages", ",".join(additional_packages)) \
#     .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
#     .config("spark.kryo.registrator", "org.apache.sedona.core.serde.SedonaKryoRegistrator") \
#     .config("spark.sql.extensions", "org.apache.sedona.sql.SedonaSqlExtensions,org.apache.sedona.viz.sql.SedonaVizExtensions") \
#     .config("spark.jars", r"D:\Artem\Work\amtech_projects\postgresql-42.7.13.jar") \
#     .master("local[*]") \
#     .getOrCreate()

    # .config("spark.executor.cores", "4") \
    # .config("spark.executor.instances", "2") \
    # .config("spark.task.cpus", "1") \
    # 
# spark = (
#     SparkSession.builder
#     .appName("my-sedona-job")
#     .master("local[*]")              # для локального запуска: [*] = все ядра, [4] = ровно 4 ядра
#     .config("spark.executor.cores", "4")      # сколько ядер у каждого executor
#     .config("spark.executor.instances", "2")  # сколько executors (опционально)
#     .config("spark.task.cpus", "1")           # сколько CPU на одну задачу (task)
#     .getOrCreate()
# )

# Создаем контекст Sedona
sedona = SedonaContext.create(config)

# print("✅ Sedona context created successfully!")
# print(f"Spark version: {spark.version}")
# 
# # Проверяем версию Sedona
# try:
#     sedona_version = spark.sparkContext._jvm.org.apache.sedona.core.utils.SedonaConstants.SEDONA_VERSION
#     print(f"Sedona version: {sedona_version}")
# except:
#     print("Sedona version not found")

# Проверка версий

In [2]:
from pyspark.sql import SparkSession

# Создаем тестовую сессию
spark = SparkSession.builder.appName("VersionCheck").master("local[*]").getOrCreate()

# 1. Версия Spark
print(f"Spark version: {spark.version}")

# 2. Версия Scala
scala_version = spark.sparkContext._jvm.scala.util.Properties.versionString()
print(f"Scala version: {scala_version}")

# 3. Версия Hadoop
hadoop_version = spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
print(f"Hadoop version: {hadoop_version}")

# 4. Версия Sedona (если уже создан контекст)
# После создания sedona = SedonaContext.create(config)
try:
    sedona_version = spark.sparkContext._jvm.org.apache.sedona.core.utils.SedonaConstants.SEDONA_VERSION
    print(f"Sedona version: {sedona.version} {sedona_version}")
except:
    print("Sedona context not yet created or SedonaConstants not found")

# spark.stop()

Spark version: 3.5.4
Scala version: version 2.12.18
Hadoop version: 3.3.4
Sedona version: 3.5.4 <py4j.java_gateway.JavaPackage object at 0x000000003F1B3850>


In [3]:
import os
import subprocess

# Проверка JAVA_HOME
print(f"JAVA_HOME: {os.environ.get('JAVA_HOME', 'NOT SET')}")

# Проверка Java из Python
try:
    result = subprocess.run(['java', '-version'], capture_output=True, text=True)
    print("Java version:", result.stderr)
except Exception as e:
    print(f"Error running java: {e}")

JAVA_HOME: C:\Users\user\SDKMAN~1\CANDID~1\java\current
Java version: java version "1.8.0_211"
Java(TM) SE Runtime Environment (build 1.8.0_211-b12)
Java HotSpot(TM) 64-Bit Server VM (build 25.211-b12, mixed mode)



# Тестирование

In [4]:
wkt = "POINT(1 1)"
sedona.sql(
 f"SELECT ST_GeomFromWKT('{wkt}') AS geom"
).show()


+-----------+
|       geom|
+-----------+
|POINT (1 1)|
+-----------+



In [5]:
sql = """
SELECT ST_AreaSpheroid(
    ST_GeomFromWKT('Polygon ((34 35, 28 30, 25 34, 34 35))')
) as result
"""
sedona.sql(sql).show(truncate=False)

+---------------------+
|result               |
+---------------------+
|2.0182485081176245E11|
+---------------------+



In [6]:
from pyspark.sql import Row
data = [
    Row(id=1, name="Point A", lat=40.7128, lon=-74.0060),
    Row(id=2, name="Point B", lat=34.0522, lon=-118.2437),
    Row(id=3, name="Point C", lat=37.7749, lon=-122.4194)
]
df = sedona.createDataFrame(data)
df.show()

+---+-------+-------+---------+
| id|   name|    lat|      lon|
+---+-------+-------+---------+
|  1|Point A|40.7128|  -74.006|
|  2|Point B|34.0522|-118.2437|
|  3|Point C|37.7749|-122.4194|
+---+-------+-------+---------+



In [7]:
cities_df = sedona \
 .createDataFrame(
 [
 ("San Francisco", -122.4191, 37.7749),
 ("New York", -74.0060, 40.7128),
 ("Austin", -97.7431, 30.2672)
 ],
 ["city", "longitude", "latitude"]
 )
cities_df.show()

+-------------+---------+--------+
|         city|longitude|latitude|
+-------------+---------+--------+
|San Francisco|-122.4191| 37.7749|
|     New York|  -74.006| 40.7128|
|       Austin| -97.7431| 30.2672|
+-------------+---------+--------+



In [ ]:
cities_df.printSchema()

In [9]:
cities_df.createOrReplaceTempView("cities")

In [10]:
cities_df = sedona \
 .sql("""
 SELECT
 *,
 ST_Point(longitude, latitude) AS geometry
 FROM cities
 """)
cities_df.show(truncate=False)

+-------------+---------+--------+-------------------------+
|city         |longitude|latitude|geometry                 |
+-------------+---------+--------+-------------------------+
|San Francisco|-122.4191|37.7749 |POINT (-122.4191 37.7749)|
|New York     |-74.006  |40.7128 |POINT (-74.006 40.7128)  |
|Austin       |-97.7431 |30.2672 |POINT (-97.7431 30.2672) |
+-------------+---------+--------+-------------------------+



In [11]:
cities_df.printSchema()

root
 |-- city: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- geometry: geometry (nullable = true)



In [12]:
cities_df.createOrReplaceTempView("cities")

In [13]:
buffer_df = sedona \
 .sql("""
 SELECT
 city,
 ST_Buffer(geometry, 1000, true) AS geometry
 FROM cities
 """)
buffer_df.show()

+-------------+--------------------+
|         city|            geometry|
+-------------+--------------------+
|San Francisco|POLYGON ((-122.40...|
|     New York|POLYGON ((-73.994...|
|       Austin|POLYGON ((-97.732...|
+-------------+--------------------+



# Загрузка файла geoparquete

In [14]:
geoparquet_file_path = r'D:\Artem\Work\amtech_projects\sedona_geoservice\data\features_plain.geoparquet'

In [15]:
loaded_df = sedona.read.format("geoparquet").load(geoparquet_file_path)

In [16]:
loaded_df.show(truncate=False)

+-------+--------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [17]:
# new_df = loaded_df.selectExpr("ST_Area(ST_Transform(ST_SetSRID(geometry, 4326), 'EPSG:3857')) as area")
new_df = loaded_df.selectExpr("ST_AreaSpheroid(geometry) as area")


In [ ]:
new_df.show()

# Подключение к PostGIS

In [7]:
credentials = dict({"host": "10.6.81.133", "port":"5432", "user":"postgres", "password":"tpY7H&sdvsdfsdf7zx9J"})
credentials

{'host': '10.6.81.133',
 'port': '5432',
 'user': 'postgres',
 'password': 'tpY7H&sdvsdfsdf7zx9J'}

In [8]:
postgresql_url = f"jdbc:postgresql://{credentials.get('host')}:{credentials.get('port')}/gisdb_8411_250226"

# Тестируем на вычислении площади. Возможны ошибки с тем, что некорректно указана SRID для объектов в таблице public.features_plain 

In [9]:
import time

# Используем ST_Area прямо в SQL запросе
query = """
    SELECT 
        id,
        layer_id,
        egip.ST_Area(egip.ST_Transform(geometry, 3857)) as area
    FROM public.features_plain 
    -- LIMIT 10000
"""

query = """
    SELECT 
        id,
        layer_id,
        geometry,
        egip.ST_Area(geometry) as area
    FROM public.features_plain 
    -- LIMIT 10000
"""
# Начинаем замер времени
start_time = time.time()
df = sedona.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", f"{credentials.get('user')}") \
    .option("password", f"{credentials.get('password')}") \
    .option("dbtable", f"({query}) as subquery") \
    .option("driver", "org.postgresql.Driver") \
    .option("numPartitions", "24") \
    .load()
# Замеряем время выполнения загрузки
load_time = time.time() - start_time
print(f"Время загрузки данных: {load_time:.2f} секунд")
# df.show(truncate=True)

Время загрузки данных: 0.85 секунд


In [10]:
df.count()

451083

In [11]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- layer_id: long (nullable = true)
 |-- geometry: string (nullable = true)
 |-- area: double (nullable = true)



In [12]:
df.show()

+-------+--------+--------------------+----+
|     id|layer_id|            geometry|area|
+-------+--------+--------------------+----+
|3441831|     164|0101000020E610000...| 0.0|
|3441832|     164|0101000020E610000...| 0.0|
|3441833|     164|0101000020E610000...| 0.0|
|3441834|     164|0101000020E610000...| 0.0|
|3441835|     164|0101000020E610000...| 0.0|
|3441836|     164|0101000020E610000...| 0.0|
|3441837|     164|0101000020E610000...| 0.0|
|3441838|     164|0101000020E610000...| 0.0|
|3441839|     164|0101000020E610000...| 0.0|
|3441840|     164|0101000020E610000...| 0.0|
|3441841|     164|0101000020E610000...| 0.0|
|3441842|     164|0101000020E610000...| 0.0|
|3441843|     164|0101000020E610000...| 0.0|
|3441844|     164|0101000020E610000...| 0.0|
|3441845|     164|0101000020E610000...| 0.0|
|3441846|     164|0101000020E610000...| 0.0|
|3441847|     164|0101000020E610000...| 0.0|
|3441848|     164|0101000020E610000...| 0.0|
|3441849|     164|0101000020E610000...| 0.0|
|3441850| 

# Приводим колонку geometry->string к типу GEOMETRY

In [7]:
# Создаем временное представление вашего DataFrame
df.createOrReplaceTempView("input_table")

# Выполняем SQL-запрос с функцией ST_GeomFromText
df = sedona.sql("""
    with subquery as (
     SELECT 
        id, layer_id, area, 
        ST_GeomFromWKB(geometry) as geometry 
    FROM input_table
    )
    SELECT * FROM subquery
    ORDER BY ST_GeoHash(geometry, 12)
    LIMIT 100
    
""")

In [8]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- layer_id: long (nullable = true)
 |-- area: double (nullable = true)
 |-- geometry: geometry (nullable = true)



In [9]:
# сохраняем результат в файл
df.write \
    .format("geoparquet") \
    .mode("overwrite") \
    .save(r"D:\Artem\Work\amtech_projects\sedona_geoservice\data\area.geoparquet")

# df.limit(100).write \
#     .format("geoparquet") \
#     .mode("overwrite") \
#     .option("geometry", "geometry") \
#     .save(r"D:\Artem\Work\amtech_projects\sedona_geoservice\data\area.geoparquet")

NameError: name 'df' is not defined

In [9]:
df.take(10)

[Row(id=3724141, layer_id=170, area=6.79783493526394e-08, geometry=<MULTIPOLYGON (((36.993 55.184, 36.993 55.184, 36.993 55.184, 36.993 55.184,...>),
 Row(id=3781881, layer_id=173, area=6.79783493526394e-08, geometry=<MULTIPOLYGON (((36.993 55.184, 36.993 55.184, 36.993 55.184, 36.993 55.184,...>),
 Row(id=3811204, layer_id=173, area=6.79783493526394e-08, geometry=<MULTIPOLYGON (((36.993 55.184, 36.993 55.184, 36.993 55.184, 36.993 55.184,...>),
 Row(id=3694818, layer_id=170, area=6.79783493526394e-08, geometry=<MULTIPOLYGON (((36.993 55.184, 36.993 55.184, 36.993 55.184, 36.993 55.184,...>),
 Row(id=3727525, layer_id=170, area=2.771353487496294e-07, geometry=<MULTIPOLYGON (((36.996 55.185, 36.996 55.185, 36.996 55.184, 36.996 55.184,...>),
 Row(id=3779551, layer_id=173, area=2.771353487496294e-07, geometry=<MULTIPOLYGON (((36.996 55.184, 36.996 55.184, 36.996 55.184, 36.996 55.184,...>),
 Row(id=3814588, layer_id=173, area=2.771353487496294e-07, geometry=<MULTIPOLYGON (((36.996 55.185

# Тестируем на кластеризации

In [9]:
postgresql_url_with_schema = f"{postgresql_url}?currentSchema=egip"
query = """
   select
	fp.id,
	fp.layer_id,
    t.district_id,
    t.region_id,
    t.district_part_id,
    t.region_part_id,
    t.original_hex_id_8,
    t.original_hex_id_9,
    t.original_hex_id_10,
    t.bordered_hex_id_8,
    t.bordered_hex_id_9,
    t.bordered_hex_id_10,
    t.custom_hex_id_8,
    t.custom_hex_id_9,
    t.custom_hex_id_10
FROM public.features_plain fp,
LATERAL h3.cluster_one_object_by_geom(ST_SetSRID(fp.geometry, 4326)) AS t
--LIMIT 10000
"""
# Начинаем замер времени
start_time = time.time()
df = sedona.read \
    .format("jdbc") \
    .option("url", postgresql_url_with_schema) \
    .option("user", f"{credentials.get('user')}") \
    .option("password", f"{credentials.get('password')}") \
    .option("dbtable", f"({query}) as subquery") \
    .option("driver", "org.postgresql.Driver") \
    .load()
# Замеряем время выполнения загрузки
load_time = time.time() - start_time
print(f"Время загрузки данных: {load_time:.2f} секунд")
# df.show(truncate=True)

Время загрузки данных: 0.25 секунд


In [10]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- layer_id: long (nullable = true)
 |-- district_id: string (nullable = true)
 |-- region_id: string (nullable = true)
 |-- district_part_id: integer (nullable = true)
 |-- region_part_id: integer (nullable = true)
 |-- original_hex_id_8: string (nullable = true)
 |-- original_hex_id_9: string (nullable = true)
 |-- original_hex_id_10: string (nullable = true)
 |-- bordered_hex_id_8: string (nullable = true)
 |-- bordered_hex_id_9: string (nullable = true)
 |-- bordered_hex_id_10: string (nullable = true)
 |-- custom_hex_id_8: string (nullable = true)
 |-- custom_hex_id_9: string (nullable = true)
 |-- custom_hex_id_10: string (nullable = true)



In [ ]:
df.take(10)

In [6]:
df.show(10, truncate=True)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "D:\Artem\Work\amtech_projects\sedona_geoservice\.venv\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\Artem\Work\amtech_projects\sedona_geoservice\.venv\Lib\site-packages\py4j\clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Roaming\uv\python\cpython-3.11.15-windows-x86_64-none\Lib\socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
df.count()

In [43]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- layer_id: long (nullable = true)
 |-- district_id: string (nullable = true)
 |-- region_id: string (nullable = true)
 |-- district_part_id: integer (nullable = true)
 |-- region_part_id: integer (nullable = true)
 |-- original_hex_id_8: string (nullable = true)
 |-- original_hex_id_9: string (nullable = true)
 |-- original_hex_id_10: string (nullable = true)
 |-- bordered_hex_id_8: string (nullable = true)
 |-- bordered_hex_id_9: string (nullable = true)
 |-- bordered_hex_id_10: string (nullable = true)
 |-- custom_hex_id_8: string (nullable = true)
 |-- custom_hex_id_9: string (nullable = true)
 |-- custom_hex_id_10: string (nullable = true)



In [18]:
# Создаем временное представление
df.createOrReplaceTempView("temp_table")

# Выполняем SQL запрос
df = sedona.sql("""
    SELECT * 
    FROM temp_table 
    LIMIT 100
""")

# Показываем результат
# df_result.show()

In [19]:
df.count()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "D:\Artem\Work\amtech_projects\sedona_geoservice\.venv\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\Artem\Work\amtech_projects\sedona_geoservice\.venv\Lib\site-packages\py4j\clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Roaming\uv\python\cpython-3.11.15-windows-x86_64-none\Lib\socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [13]:
# Берем первые 100 строк и сохраняем в Parquet
df.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(r"D:\Artem\Work\amtech_projects\sedona_geoservice\data\clusterization.parquet")
# Берем первые 100 строк и сохраняем в Parquet
# df.limit(100).write \
#     .format("parquet") \
#     .mode("overwrite") \
#     .save(r"D:\Artem\Work\amtech_projects\sedona_geoservice\data\clusterization.parquet")

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "D:\Artem\Work\amtech_projects\sedona_geoservice\.venv\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\Artem\Work\amtech_projects\sedona_geoservice\.venv\Lib\site-packages\py4j\clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Roaming\uv\python\cpython-3.11.15-windows-x86_64-none\Lib\socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

# Тестируем на удалённом кластере

## Площадь

In [26]:
sedona = SedonaContext.create(config)
df = sedona.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", credentials['user']) \
    .option("password", credentials['password']) \
    .option("dbtable", "public.features_plain") \
    .option("driver", "org.postgresql.Driver") \
    .load()

In [27]:
df.show()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "D:\Artem\Work\amtech_projects\sedona_geoservice\.venv\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\Artem\Work\amtech_projects\sedona_geoservice\.venv\Lib\site-packages\py4j\clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Roaming\uv\python\cpython-3.11.15-windows-x86_64-none\Lib\socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

# TODO
## Добавить фильтры на чтение
## Проверить типы, геометрии полей

In [2]:
# Используем ST_Area прямо в SQL запросе
query = """
    SELECT 
        ST_Area(ST_Transfrorm(geometry, 3857)) as area,
        geometry
    FROM public.features_plain 
    LIMIT 1000
"""

df = sedona.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", f"{credentials.get('user')}") \
    .option("password", f"{credentials.get('password')}") \
    .option("dbtable", f"({query}) as subquery") \
    .option("driver", "org.postgresql.Driver") \
    .load()

df.show()

NameError: name 'postgresql_url' is not defined

In [24]:
table_name = "public.features_plain"
df = sedona.read \
 .format("jdbc") \
 .option("url", postgresql_url) \
 .option("user",  f"{credentials.get('user')}") \
 .option("password", f"{credentials.get('password')}") \
 .option("dbtable", "public.features_plain") \
 .option("driver", "org.postgresql.Driver") \
 .load()
df.show()


Py4JJavaError: An error occurred while calling o262.load.
: org.postgresql.util.PSQLException: FATAL: database "sedona" does not exist
	at org.postgresql.core.v3.QueryExecutorImpl.receiveErrorResponse(QueryExecutorImpl.java:2993)
	at org.postgresql.core.v3.QueryExecutorImpl.readStartupMessages(QueryExecutorImpl.java:3118)
	at org.postgresql.core.v3.QueryExecutorImpl.<init>(QueryExecutorImpl.java:251)
	at org.postgresql.core.v3.ConnectionFactoryImpl.openConnectionImpl(ConnectionFactoryImpl.java:398)
	at org.postgresql.core.ConnectionFactory.openConnection(ConnectionFactory.java:52)
	at org.postgresql.jdbc.PgConnection.<init>(PgConnection.java:295)
	at org.postgresql.Driver.makeConnection(Driver.java:437)
	at org.postgresql.Driver.connect(Driver.java:305)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:49)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.creat

In [40]:
df = sedona.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", credentials['user']) \
    .option("password", credentials['password']) \
    .option("dbtable", "public.features_plain") \
    .load()  # Без явного driver

Py4JJavaError: An error occurred while calling o448.load.
: java.sql.SQLException: No suitable driver
	at java.sql/java.sql.DriverManager.getDriver(DriverManager.java:299)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$2(JDBCOptions.scala:109)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:109)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:41)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcRelationProvider.createRelation(JdbcRelationProvider.scala:34)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:346)
	at org.apache.spark.sql.DataFrameReader.loadV1Source(DataFrameReader.scala:229)
	at org.apache.spark.sql.DataFrameReader.$anonfun$load$2(DataFrameReader.scala:211)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.sql.DataFrameReader.load(Data